# RoadWise — Experiment A: D20 Alligator Crack Error Analysis
### Systematic Evaluation & Error Diagnosis on Held-Out Test Set (522 Images)

**Context:**
- **Trained Model:** YOLOv8s from run `yolov8s_3class_final_run_20261110_212046`
- **Checkpoint:** `/content/drive/MyDrive/RoadWise_Training/yolov8s_3class_final_run_20261110_212046/weights/best.pt`
- **Test Set:** 522 images, 919 ground-truth objects (D00=240, D10=318, D20=361)
- **Goal:** Diagnose D20 recall deficit (14.7% recall, 15.63% mAP50) via class-aware IoU matching and visual galleries.

## 1. Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
import os, sys, shutil, zipfile

drive.mount('/content/drive')

CHECKPOINT_PATH = '/content/drive/MyDrive/RoadWise_Training/yolov8s_3class_final_run_20261110_212046/weights/best.pt'
assert os.path.exists(CHECKPOINT_PATH), f'CRITICAL GATE FAILURE: Checkpoint not found at {CHECKPOINT_PATH}'
print(f'✅ Verified Experiment A Checkpoint: {CHECKPOINT_PATH}')

## 2. Install Dependencies

In [ ]:
!pip install -q ultralytics pyyaml pillow numpy matplotlib
import ultralytics
ultralytics.checks()

## 3. Unpack Experiment A Dataset & Verify Split Integrity

In [ ]:
zip_candidates = [
    '/content/crack_experiment_a_3class.zip',
    '/content/drive/MyDrive/RoadWise_Training/crack_experiment_a_3class.zip',
    '/content/drive/MyDrive/RoadWise/crack_experiment_a_3class.zip',
    '/content/drive/MyDrive/crack_experiment_a_3class.zip'
]

dataset_zip = None
for zc in zip_candidates:
    if os.path.exists(zc):
        dataset_zip = zc
        break

if not dataset_zip:
    from google.colab import files
    print('Please upload crack_experiment_a_3class.zip:')
    uploaded = files.upload()
    dataset_zip = list(uploaded.keys())[0]

print(f'Unpacking {dataset_zip}...')
with zipfile.ZipFile(dataset_zip, 'r') as z:
    z.extractall('/content/')

TEST_IMG_DIR = '/content/crack_experiment_a_3class/test/images'
TEST_LBL_DIR = '/content/crack_experiment_a_3class/test/labels'

test_imgs = [f for f in os.listdir(TEST_IMG_DIR) if f.lower().endswith(('.jpg', '.png'))]
test_lbls = [f for f in os.listdir(TEST_LBL_DIR) if f.endswith('.txt')]
assert len(test_imgs) == 522, f'Expected 522 test images, found {len(test_imgs)}'
assert len(test_lbls) == 522, f'Expected 522 test labels, found {len(test_lbls)}'
print('✅ Verified Test Split: 522 images and 522 labels')

## 4. Run D20 Error Analysis & Matching Workflow

In [ ]:
import json
from collections import Counter, defaultdict
import numpy as np
from PIL import Image, ImageDraw
from ultralytics import YOLO

OUTPUT_DIR = '/content/d20_error_analysis_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = {0: 'longitudinal_crack', 1: 'transverse_crack', 2: 'alligator_crack'}
CLASS_SHORT = {0: 'D00', 1: 'D10', 2: 'D20'}
COLORS = {'gt': (0, 200, 0), 'd00': (30, 144, 255), 'd10': (255, 165, 0), 'd20': (220, 20, 60)}

def box_iou_xyxy(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area1 = max(0.0, box1[2] - box1[0]) * max(0.0, box1[3] - box1[1])
    area2 = max(0.0, box2[2] - box2[0]) * max(0.0, box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

def yolo_to_xyxy(box_yolo, img_w, img_h):
    xc, yc, w, h = box_yolo
    return [(xc - w / 2.0) * img_w, (yc - h / 2.0) * img_h, (xc + w / 2.0) * img_w, (yc + h / 2.0) * img_h]

# Load Ground Truth
gt_data = {}
gt_counts = Counter()
for lf in sorted(test_lbls):
    stem = os.path.splitext(lf)[0]
    img_f = stem + '.jpg' if os.path.exists(os.path.join(TEST_IMG_DIR, stem + '.jpg')) else stem + '.png'
    with Image.open(os.path.join(TEST_IMG_DIR, img_f)) as img:
        iw, ih = img.size
    boxes = []
    with open(os.path.join(TEST_LBL_DIR, lf)) as f:
        for line in f:
            p = line.strip().split()
            if not p: continue
            cid = int(p[0])
            byolo = [float(x) for x in p[1:5]]
            bxy = yolo_to_xyxy(byolo, iw, ih)
            boxes.append({'class_id': cid, 'box_xyxy': bxy, 'area': (bxy[2]-bxy[0])*(bxy[3]-bxy[1])})
            gt_counts[cid] += 1
    gt_data[img_f] = boxes

print(f'Ground Truth Counts: D00={gt_counts[0]}, D10={gt_counts[1]}, D20={gt_counts[2]} (Total: {sum(gt_counts.values())})')
assert sum(gt_counts.values()) == 919 and gt_counts[0] == 240 and gt_counts[1] == 318 and gt_counts[2] == 361

# Load YOLO Model & Run Inference
model = YOLO(CHECKPOINT_PATH)
print('Running inference across all 522 test images at imgsz=640...')
raw_preds = {}
for img_f in sorted(gt_data.keys()):
    res = model.predict(source=os.path.join(TEST_IMG_DIR, img_f), conf=0.05, imgsz=640, verbose=False)
    plist = []
    if len(res) > 0 and res[0].boxes is not None:
        bobj = res[0].boxes
        for i in range(len(bobj)):
            plist.append({'class_id': int(bobj.cls[i].item()), 'confidence': float(bobj.conf[i].item()), 'box_xyxy': bobj.xyxy[i].tolist()})
    raw_preds[img_f] = plist

# Multi-threshold evaluation
def evaluate_matching(conf_thresh, iou_thresh):
    tp, fn, fp = Counter(), Counter(), Counter()
    confusion = defaultdict(Counter)
    d20_details = {'tp': 0, 'fn_missed': 0, 'confused_d00': 0, 'confused_d10': 0, 'fp': 0}
    for img_f, gboxes in gt_data.items():
        pboxes = [p for p in raw_preds.get(img_f, []) if p['confidence'] >= conf_thresh]
        matched_g, matched_p = set(), set()
        # Pass 1: True class matches
        cand = []
        for gi, g in enumerate(gboxes):
            for pi, p in enumerate(pboxes):
                if g['class_id'] == p['class_id']:
                    iou = box_iou_xyxy(g['box_xyxy'], p['box_xyxy'])
                    if iou >= iou_thresh: cand.append((iou, gi, pi))
        cand.sort(reverse=True)
        for iou, gi, pi in cand:
            if gi not in matched_g and pi not in matched_p:
                matched_g.add(gi); matched_p.add(pi)
                tp[gboxes[gi]['class_id']] += 1
                if gboxes[gi]['class_id'] == 2: d20_details['tp'] += 1
        # Pass 2: Cross class confusion
        cross = []
        for gi, g in enumerate(gboxes):
            if gi in matched_g: continue
            for pi, p in enumerate(pboxes):
                if pi in matched_p: continue
                iou = box_iou_xyxy(g['box_xyxy'], p['box_xyxy'])
                if iou >= iou_thresh: cross.append((iou, gi, pi))
        cross.sort(reverse=True)
        for iou, gi, pi in cross:
            if gi not in matched_g and pi not in matched_p:
                matched_g.add(gi); matched_p.add(pi)
                g_cls, p_cls = gboxes[gi]['class_id'], pboxes[pi]['class_id']
                confusion[g_cls][p_cls] += 1
                fn[g_cls] += 1; fp[p_cls] += 1
                if g_cls == 2:
                    if p_cls == 0: d20_details['confused_d00'] += 1
                    elif p_cls == 1: d20_details['confused_d10'] += 1
        # Unmatched
        for gi in range(len(gboxes)):
            if gi not in matched_g:
                fn[gboxes[gi]['class_id']] += 1
                if gboxes[gi]['class_id'] == 2: d20_details['fn_missed'] += 1
        for pi in range(len(pboxes)):
            if pi not in matched_p:
                fp[pboxes[pi]['class_id']] += 1
                if pboxes[pi]['class_id'] == 2: d20_details['fp'] += 1
    return {'tp': dict(tp), 'fn': dict(fn), 'fp': dict(fp), 'd20': d20_details}

print('\n' + '='*60)
print('D20 ERROR ANALYSIS RESULTS ACROSS CONFIDENCE THRESHOLDS (IoU=0.50)')
print('='*60)
for c in [0.10, 0.25, 0.50]:
    r = evaluate_matching(c, 0.50)
    d = r['d20']
    total_gt = gt_counts[2]
    recall = d['tp'] / total_gt
    prec = d['tp'] / max(1, (d['tp'] + d['fp']))
    print(f'Conf={c:.2f} | D20 TP={d["tp"]:3d} | Missed={d["fn_missed"]:3d} | Confused(D00)={d["confused_d00"]:2d} | Confused(D10)={d["confused_d10"]:2d} | FP={d["fp"]:3d} | Recall={recall:.3f} | Precision={prec:.3f}')

## 5. Sync Results & Artifacts to Google Drive

In [ ]:
DRIVE_DEST = '/content/drive/MyDrive/RoadWise_Training/yolov8s_3class_final_run_20261110_212046/d20_error_analysis'
os.makedirs(DRIVE_DEST, exist_ok=True)
print(f'Syncing results to: {DRIVE_DEST} ...')
shutil.copytree(OUTPUT_DIR, DRIVE_DEST, dirs_exist_ok=True)
print('✅ Analysis artifacts backed up to Google Drive.')